# Capítulo 32 — Organizar resúmenes con tablas dinámicas

## De agrupar datos a cruzar variables

En los capítulos anteriores usamos `groupby()` para resumir datos por grupos.

Agrupamos por día, por horario y también por más de una variable a la vez. Eso nos permitió responder preguntas como:

- ¿qué día tuvo mayor facturación total?
- ¿qué horario tuvo mayor promedio de cuenta?
- ¿qué combinaciones de día y horario aparecen en el dataset?

Ahora vamos a trabajar con otra forma de organizar resúmenes: las **tablas dinámicas**.

Una tabla dinámica permite cruzar variables. Por ejemplo, podemos colocar los días en las filas, los horarios en las columnas y dentro de la tabla mostrar un indicador, como el promedio de cuenta o la cantidad de registros.

Esto no reemplaza a `groupby()`. Es otra forma de resumir datos, especialmente útil cuando queremos comparar categorías en forma de tabla.

## Cargar el dataset de trabajo

Vamos a seguir usando el dataset `tips` de Seaborn. Como ya conocemos sus columnas principales, podemos concentrarnos en la forma de organizar los resúmenes.

In [ ]:
import pandas as pd
import seaborn as sns

df = sns.load_dataset("tips")

df.head()

,total_bill,tip,sex,smoker,day,time,size
0,16.99,1.01,Female,No,Sun,Dinner,2
1,10.34,1.66,Male,No,Sun,Dinner,3
2,21.01,3.50,Male,No,Sun,Dinner,3
3,23.68,3.31,Male,No,Sun,Dinner,2
4,24.59,3.61,Female,No,Sun,Dinner,4


La tabla muestra las primeras filas del dataset.

Seguimos trabajando con las columnas que ya conocemos: `total_bill`, `tip`, `day`, `time`, `size` y otras variables categóricas.

En este capítulo nos van a interesar especialmente las columnas `day` y `time`, porque nos permiten cruzar dos categorías: el día y el horario. También vamos a usar columnas numéricas como `total_bill` y `tip`, que serán los valores que vamos a resumir dentro de las tablas dinámicas.

## Recordar una agrupación doble con `groupby()`

Antes de crear una tabla dinámica, vamos a recordar cómo se veía una agrupación doble con `groupby()`. La pregunta será:

**¿Cuál es el promedio de cuenta según el día y el horario?**

Primero responderemos esta pregunta con `groupby()`.

In [ ]:
promedio_groupby = df.groupby(
    ["day", "time"],
    observed=True
)["total_bill"].mean().round(2)

promedio_groupby

day   time  
Thur  Lunch     17.66
      Dinner    18.78
Fri   Lunch     12.85
      Dinner    19.66
Sat   Dinner    20.44
Sun   Dinner    21.41
Name: total_bill, dtype: float64

La salida muestra el promedio de cuenta para cada combinación de `day` y `time`.

Por ejemplo:

- `Thur` en `Lunch` tiene un promedio de cuenta de **17.66**.
- `Thur` en `Dinner` tiene un promedio de cuenta de **18.78**.
- `Fri` en `Lunch` tiene un promedio de cuenta de **12.85**.
- `Fri` en `Dinner` tiene un promedio de cuenta de **19.66**.

Esta salida es correcta y útil, pero tiene forma de lista agrupada.

Para comparar visualmente días y horarios, puede ser más cómodo reorganizar el resultado en forma de tabla, colocando los días como filas y los horarios como columnas.

## Crear una tabla dinámica con `pivot_table()`

Vamos a responder la misma pregunta, pero usando una tabla dinámica. La pregunta sigue siendo:

**¿Cuál es el promedio de cuenta según el día y el horario?**

En esta tabla vamos a colocar:

- los días en las filas;
- los horarios en las columnas;
- el promedio de cuenta dentro de la tabla.

In [ ]:
tabla_promedio_cuenta = df.pivot_table(
    values="total_bill",
    index="day",
    columns="time",
    aggfunc="mean",
    observed=True
).round(2)

tabla_promedio_cuenta

time,Lunch,Dinner
day,,
Thur,17.66,18.78
Fri,12.85,19.66
Sat,NaN,20.44
Sun,NaN,21.41


La tabla dinámica muestra la misma información que obtuvimos con `groupby()`, pero organizada de otra manera. Ahora cada fila representa un día y cada columna representa un horario.

Esto facilita algunas comparaciones. Por ejemplo, podemos ver rápidamente que:

- `Thur` tiene datos tanto para `Lunch` como para `Dinner`;
- `Fri` también tiene datos para ambos horarios;
- `Sat` y `Sun` solo muestran datos para `Dinner`.

También aparecen algunos valores `NaN`. Por ahora podemos interpretarlos como combinaciones sin datos, y más adelante los vamos a revisar con más detalle.

## Entender los parámetros principales

En la tabla dinámica anterior usamos cuatro parámetros importantes:

- `values`: indica qué columna numérica queremos resumir.
- `index`: indica qué variable se ubicará en las filas.
- `columns`: indica qué variable se ubicará en las columnas.
- `aggfunc`: indica qué cálculo se aplicará sobre los datos.

En nuestro ejemplo:

- `values="total_bill"` significa que resumimos la columna de cuentas;
- `index="day"` coloca los días en las filas;
- `columns="time"` coloca los horarios en las columnas;
- `aggfunc="mean"` calcula el promedio.

También usamos `observed=True` porque algunas columnas categóricas, como `day` y `time`, están guardadas como categorías.

En este caso, este parámetro ayuda a trabajar con las combinaciones observadas en los datos y evita avisos de Pandas relacionados con cambios futuros.

## Tabla dinámica de cantidad de cuentas

Una tabla dinámica no sirve solamente para calcular promedios. También podemos contar cuántos registros hay en cada combinación de categorías. Ahora vamos a responder esta pregunta:

**¿Cuántas cuentas hay para cada combinación de día y horario?**

In [ ]:
tabla_cantidad = df.pivot_table(
    values="total_bill",
    index="day",
    columns="time",
    aggfunc="count",
    observed=True
)

tabla_cantidad

time,Lunch,Dinner
day,,
Thur,61.0,1.0
Fri,7.0,12.0
Sat,NaN,87.0
Sun,NaN,76.0


La tabla muestra cuántas cuentas hay para cada combinación de día y horario.

Por ejemplo:

- `Thur` tiene **61** cuentas en `Lunch` y **1** cuenta en `Dinner`.
- `Fri` tiene **7** cuentas en `Lunch` y **12** en `Dinner`.
- `Sat` tiene **87** cuentas en `Dinner`.
- `Sun` tiene **76** cuentas en `Dinner`.

También aparecen valores `NaN` en algunas combinaciones. En este caso, esos valores indican que no hay cuentas registradas para esa combinación de día y horario.

Como estamos contando registros, esos `NaN` pueden interpretarse como ausencia de datos para esa combinación.

Además, al aparecer valores `NaN`, Pandas muestra los conteos con decimales, como `61.0` o `1.0`. Eso no cambia el significado: siguen siendo cantidades de cuentas.

## Reemplazar valores vacíos en una tabla de conteos

En una tabla de conteos, puede ser útil reemplazar los valores vacíos por `0`. Si no hay registros para una combinación, podemos mostrarlo como cero cuentas.

Para eso, `pivot_table()` permite usar el parámetro `fill_value`.

In [ ]:
tabla_cantidad = df.pivot_table(
    values="total_bill",
    index="day",
    columns="time",
    aggfunc="count",
    observed=True,
    fill_value=0
)

tabla_cantidad

time,Lunch,Dinner
day,,
Thur,61,1
Fri,7,12
Sat,0,87
Sun,0,76


Ahora la tabla muestra `0` en las combinaciones donde no había registros.

Por ejemplo:

- `Sat` en `Lunch` aparece como **0**;
- `Sun` en `Lunch` aparece como **0**.

En una tabla de conteos, esto resulta razonable: si no hay cuentas para una combinación, podemos mostrar cero cuentas.

Pero este criterio no debe aplicarse automáticamente a todos los casos. En una tabla de promedios, reemplazar un valor vacío por `0` podría ser confuso, porque podría parecer que el promedio real fue cero, cuando en realidad no había datos para calcularlo.

## Tabla dinámica de total facturado

Además de contar registros, también podemos sumar valores. Ahora vamos a calcular el total facturado para cada combinación de día y horario. La pregunta será:

**¿Cuánto se facturó en total según el día y el horario?**

In [ ]:
tabla_total_facturado = df.pivot_table(
    values="total_bill",
    index="day",
    columns="time",
    aggfunc="sum",
    observed=True
).round(2)

tabla_total_facturado

time,Lunch,Dinner
day,,
Thur,1077.55,18.78
Fri,89.92,235.96
Sat,NaN,1778.40
Sun,NaN,1627.16


La tabla muestra el total facturado para cada combinación de día y horario.

Podemos observar, por ejemplo, que:

- `Thur` en `Lunch` acumuló **1077.55**;
- `Fri` en `Dinner` acumuló **235.96**;
- `Sat` en `Dinner` acumuló **1778.40**;
- `Sun` en `Dinner` acumuló **1627.16**.

En este caso, el total facturado depende mucho de la cantidad de cuentas registradas en cada combinación.

Por eso, al interpretar esta tabla conviene recordar la tabla anterior de conteos. Una combinación puede facturar más no necesariamente porque sus cuentas sean más altas, sino porque tiene más registros.

## Tabla dinámica de promedio de propina

También podemos usar otra columna numérica como valor de análisis. Ahora vamos a calcular el promedio de propina para cada combinación de día y horario. La pregunta será:

**¿Cuál fue la propina promedio según el día y el horario?**

In [ ]:
tabla_promedio_propina = df.pivot_table(
    values="tip",
    index="day",
    columns="time",
    aggfunc="mean",
    observed=True
).round(2)

tabla_promedio_propina

time,Lunch,Dinner
day,,
Thur,2.77,3.00
Fri,2.38,2.94
Sat,NaN,2.99
Sun,NaN,3.26


La tabla muestra el promedio de propina para cada combinación de día y horario.

Podemos observar, por ejemplo, que:

- `Thur` en `Lunch` tiene una propina promedio de **2.77**;
- `Fri` en `Dinner` tiene una propina promedio de **2.94**;
- `Sat` en `Dinner` tiene una propina promedio de **2.99**;
- `Sun` en `Dinner` tiene una propina promedio de **3.26**.

En esta tabla estamos usando la columna `tip` como valor de análisis, pero mantenemos la misma organización: días en las filas y horarios en las columnas.

Esto muestra una ventaja de las tablas dinámicas: podemos conservar la misma estructura y cambiar el indicador que queremos analizar.

## Combinaciones sin datos

En varias tablas dinámicas aparecieron valores `NaN`. Ahora vamos a comprobar de dónde salen esos valores.

En este dataset no aparecen cuentas de `Sat` en `Lunch` ni de `Sun` en `Lunch`. Para verificarlo, vamos a filtrar el dataset original.

In [ ]:
df[
    (df["day"].isin(["Sat", "Sun"])) &
    (df["time"] == "Lunch")
]

,total_bill,tip,sex,smoker,day,time,size


La salida muestra solamente los nombres de las columnas, pero no contiene filas.

Esto significa que el filtro no encontró registros que cumplan ambas condiciones:

- que el día sea `Sat` o `Sun`;
- que el horario sea `Lunch`.

Por eso, en las tablas dinámicas aparecían valores `NaN` para esas combinaciones.

Este ejemplo muestra algo importante: una celda vacía en una tabla dinámica puede representar una combinación que no existe en los datos, no necesariamente un problema del dataset.

## Agregar totales a una tabla dinámica

Las tablas dinámicas también pueden incluir totales. Con el parámetro `margins=True`, Pandas agrega una fila y una columna adicional con resúmenes generales.

Vamos a calcular nuevamente el total facturado por día y horario, pero agregando totales.

In [ ]:
tabla_total_con_totales = df.pivot_table(
    values="total_bill",
    index="day",
    columns="time",
    aggfunc="sum",
    observed=True,
    margins=True
).round(2)

tabla_total_con_totales

time,Lunch,Dinner,All
day,,,
Thur,1077.55,18.78,1096.33
Fri,89.92,235.96,325.88
Sat,NaN,1778.40,1778.40
Sun,NaN,1627.16,1627.16
All,1167.47,3660.30,4827.77


La tabla ahora incluye una fila y una columna llamadas `All`.

La columna `All` muestra el total facturado por cada día, sumando `Lunch` y `Dinner`.

Por ejemplo:

- `Thur` acumuló un total de **1096.33**;
- `Fri` acumuló **325.88**;
- `Sat` acumuló **1778.40**;
- `Sun` acumuló **1627.16**.

La fila `All` muestra el total facturado por cada horario:

- `Lunch` acumuló **1167.47**;
- `Dinner` acumuló **3660.30**.

La celda final, donde se cruza `All` con `All`, muestra el total general del dataset: **4827.77**.

Es importante recordar que `All` no es una categoría original del dataset. Es una fila y una columna agregadas por Pandas para mostrar totales.

## Tablas dinámicas con más de un valor

Hasta ahora usamos una sola columna como valor de análisis. Pero `pivot_table()` también permite resumir más de una columna numérica al mismo tiempo.

Por ejemplo, podemos calcular el promedio de `total_bill` y el promedio de `tip` en una misma tabla dinámica.

In [ ]:
tabla_dos_valores = df.pivot_table(
    values=["total_bill", "tip"],
    index="day",
    columns="time",
    aggfunc="mean",
    observed=True
).round(2)

tabla_dos_valores

tip        total_bill       
time Lunch Dinner      Lunch Dinner
day                                
Thur  2.77   3.00      17.66  18.78
Fri   2.38   2.94      12.85  19.66
Sat    NaN   2.99        NaN  20.44
Sun    NaN   3.26        NaN  21.41

La tabla dinámica ahora resume dos columnas numéricas: `tip` y `total_bill`.

Para cada una de esas columnas, Pandas muestra los valores separados por horario. Por eso la tabla tiene dos niveles de encabezado:

- primero aparece la variable resumida: `tip` o `total_bill`;
- debajo aparece el horario: `Lunch` o `Dinner`.

Esta forma de tabla permite reunir más información en una sola salida, pero también puede volverse más difícil de leer.

Por eso, cuando una tabla dinámica incluye varios valores, conviene preguntarse si realmente mejora la comparación o si sería más claro mostrar tablas separadas.

## Tablas dinámicas con más de una función

También podemos aplicar más de una función sobre una misma columna. Por ejemplo, podemos resumir `total_bill` calculando al mismo tiempo:

- cantidad de cuentas;
- promedio de cuenta;
- total facturado.

Para eso, en `aggfunc` podemos pasar una lista de funciones.

In [ ]:
tabla_varias_funciones = df.pivot_table(
    values="total_bill",
    index="day",
    columns="time",
    aggfunc=["count", "mean", "sum"],
    observed=True
).round(2)

tabla_varias_funciones

count          mean             sum         
time Lunch Dinner  Lunch Dinner    Lunch   Dinner
day                                              
Thur  61.0    1.0  17.66  18.78  1077.55    18.78
Fri    7.0   12.0  12.85  19.66    89.92   235.96
Sat    NaN   87.0    NaN  20.44      NaN  1778.40
Sun    NaN   76.0    NaN  21.41      NaN  1627.16

La tabla dinámica ahora aplica tres funciones sobre la columna `total_bill`:

- `count`: cuenta cuántos registros hay;
- `mean`: calcula el promedio;
- `sum`: calcula el total.

Para cada función, Pandas vuelve a separar los resultados por horario: `Lunch` y `Dinner`.

Esta tabla concentra mucha información en una sola salida. Por ejemplo, para `Thur` en `Lunch` podemos leer:

- cantidad de cuentas: **61**;
- promedio de cuenta: **17.66**;
- total facturado: **1077.55**.

Este tipo de tabla puede ser muy útil, pero también puede volverse más difícil de leer si agregamos demasiadas variables o demasiadas funciones.

Por eso, al construir tablas dinámicas conviene equilibrar dos necesidades: resumir bastante información, pero mantener una tabla que pueda interpretarse con claridad.

## `groupby()` y `pivot_table()`

Tanto `groupby()` como `pivot_table()` permiten resumir datos por grupos. La diferencia principal está en la forma de organizar el resultado.

Con `groupby()`, solemos obtener una tabla larga o una serie agrupada. Esa forma es muy útil para calcular, ordenar, filtrar y seguir trabajando con los datos.

Con `pivot_table()`, el resultado se organiza como una tabla cruzada: una variable queda en las filas, otra en las columnas y los valores resumidos quedan dentro de la tabla.

No son herramientas rivales. Son dos formas distintas de resumir información. En general:

- `groupby()` resulta cómodo para calcular resúmenes por grupos;
- `pivot_table()` resulta cómodo para comparar categorías cruzadas en forma de tabla.

## Una tabla dinámica para comunicar

Para terminar, vamos a construir una tabla dinámica simple y fácil de leer. Mostraremos la cantidad de cuentas por día y horario, reemplazando las combinaciones sin datos por `0`.

Esta tabla resume de forma clara cómo se distribuyen los registros del dataset.

In [ ]:
tabla_final = df.pivot_table(
    values="total_bill",
    index="day",
    columns="time",
    aggfunc="count",
    observed=True,
    fill_value=0
)

tabla_final

time,Lunch,Dinner
day,,
Thur,61,1
Fri,7,12
Sat,0,87
Sun,0,76


La tabla final resume la cantidad de cuentas por día y horario.

Es una tabla simple y clara para comunicar la distribución de los registros:

- `Thur` concentra la mayoría de sus cuentas en `Lunch`;
- `Fri` tiene registros tanto en `Lunch` como en `Dinner`;
- `Sat` y `Sun` solo tienen registros en `Dinner`;
- las combinaciones sin registros aparecen como `0`.

En este caso, usar `fill_value=0` ayuda a leer mejor la tabla, porque estamos trabajando con cantidades de registros.

## Cierre del capítulo

En este capítulo trabajamos con tablas dinámicas usando `pivot_table()`.

Vimos que una tabla dinámica permite cruzar variables categóricas y resumir una columna numérica dentro de esa combinación.

Usamos los parámetros principales:

- `values`, para indicar qué columna queremos resumir;
- `index`, para definir las filas;
- `columns`, para definir las columnas;
- `aggfunc`, para elegir el cálculo que queremos aplicar.

También vimos que una tabla dinámica puede calcular distintos tipos de resumen: promedios, conteos, sumas y otros indicadores.

Aparecieron valores `NaN` en combinaciones sin datos. Aprendimos que esos valores no siempre indican un error: muchas veces muestran que no hay registros para una combinación de categorías.

También usamos `fill_value=0`, pero con cuidado. En tablas de conteo puede ser útil reemplazar valores vacíos por cero. En cambio, en tablas de promedios podría ser engañoso.

Finalmente, vimos que `pivot_table()` puede incluir totales con `margins=True`, trabajar con más de una columna de valores y aplicar más de una función de agregación.

`groupby()` y `pivot_table()` no son herramientas rivales. Ambas sirven para resumir datos, pero organizan el resultado de manera diferente.

En el próximo capítulo vamos a empezar a trabajar con gráficos desde cero. Para eso usaremos Matplotlib, una de las bibliotecas más importantes de Python para visualización de datos.